# Notebook 03: Simulation — Chạy end-to-end

**Mục tiêu học tập:**
- Hiểu vòng lặp simulation: map → A* → trajectory → MPC → log
- Trực quan hóa kết quả simulation
- So sánh MPC trên các map khác nhau

In [ ]:
import sys
sys.path.insert(0, '..')

import yaml
import numpy as np
import matplotlib.pyplot as plt

from planner.astar import astar, load_map, path_to_cells
from planner.trajectory import path_to_trajectory, smooth_trajectory
from controller.iterative_mpc import IterativeMPC
from simulator.environment import Environment
from simulator.scenarios import load_scenario
from simulator.run_simulation import run_sim

## 1. End-to-End Simulation: basic_circle

In [ ]:
# Load scenario
scenario = load_scenario('../configs/scenarios.yaml', 'basic_circle')
env = Environment(scenario['map'])
m = load_map(scenario['map'])

# Path planning
path_cells = astar(m['grid'], scenario['start'], scenario['goal'])
path_xy = path_to_cells(path_cells, m['resolution'])
traj = path_to_trajectory(path_xy, dt=0.1, target_speed=1.0)
traj = smooth_trajectory(traj, window=5)

# MPC config
with open('../configs/mpc_baseline.yaml') as f:
    mpc_config = yaml.safe_load(f)
mpc = IterativeMPC(mpc_config)

# Run simulation
log, summary = run_sim(env, mpc, traj, dt=mpc_config['dt'])

print("Summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

In [ ]:
# Plot results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Trajectory
ax = axes[0]
ax.imshow(env.grid, cmap='gray_r', origin='upper', alpha=0.3)
ax.plot(traj['x_ref'] / m['resolution'], traj['y_ref'] / m['resolution'],
        'b--', label='Reference', linewidth=2)
ax.plot(log['x'] / m['resolution'], log['y'] / m['resolution'],
        'r-', label='Actual', linewidth=1.5)
ax.legend()
ax.set_title('Trajectory Tracking')

# Error
ax = axes[1]
n = min(len(log), len(traj))
err = np.sqrt((log['x'].values[:n] - traj['x_ref'].values[:n])**2 +
              (log['y'].values[:n] - traj['y_ref'].values[:n])**2)
ax.plot(log['t'].values[:n], err, 'g-')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Error (m)')
ax.set_title('Position Error')
ax.grid(True, alpha=0.3)

# Controls
ax = axes[2]
ax.plot(log['t'], log['steer'], label='Steer')
ax.plot(log['t'], log['accel'], label='Accel')
ax.set_xlabel('Time (s)')
ax.set_title('Control Signals')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Simulation trên map có vật cản

In [ ]:
scenario = load_scenario('../configs/scenarios.yaml', 'obstacles')
env = Environment(scenario['map'])
m = load_map(scenario['map'])

path_cells = astar(m['grid'], scenario['start'], scenario['goal'])
path_xy = path_to_cells(path_cells, m['resolution'])
traj = path_to_trajectory(path_xy, dt=0.1, target_speed=1.0)
traj = smooth_trajectory(traj, window=5)

log, summary = run_sim(env, mpc, traj, dt=mpc_config['dt'])

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(env.grid, cmap='gray_r', origin='upper', alpha=0.3)
ax.plot(traj['x_ref'] / m['resolution'], traj['y_ref'] / m['resolution'],
        'b--', label='Reference', linewidth=2)
ax.plot(log['x'] / m['resolution'], log['y'] / m['resolution'],
        'r-', label='Actual', linewidth=1.5)
ax.legend()
ax.set_title(f"Obstacles — Collisions: {summary['collision_count']}")
plt.show()

## Tóm tắt

- Vòng lặp simulation kết nối planner → controller → environment
- Simulation log ghi lại toàn bộ state và control tại mỗi bước
- Collision detection dựa trên grid map

**Bài tập:** Chạy simulation trên corridor map, điều chỉnh `target_speed` và quan sát.